In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# The Rules

In second-order-odes.ipynb we met numpy's rules for combining arrays of different shapes, called *broadcasting*:

1. Line the shapes up **from the right** (starting with the last axis).
2. Two axes are compatible if they're **equal**, or if one of them is **1**.  An axis of length 1 gets stretched to match the other.
3. If one shape has fewer axes, pretend it has extra axes of length 1 **on the left**.

The result has the larger length along every axis.  And when the shapes don't fit, we can add an axis of length 1 wherever we like with `np.newaxis`.

This assignment is practice with those rules.  No physics today!

# Predict the Shape

For each pair of shapes, predict the shape you get by combining them (with `+`, say), or say that they're incompatible.  Do it by hand, with the rules, *before* you run the cell below!

| | shape A | shape B | result |
|---|---|---|---|
| a | `(3,)` | `()` | |
| b | `(4, 3)` | `(3,)` | |
| c | `(4, 3)` | `(4,)` | |
| d | `(4, 1)` | `(3,)` | |
| e | `(2, 1, 5)` | `(3, 1)` | |
| f | `(2, 5)` | `(5, 2)` | |
| g | `(1000, 2, 1)` | `(2, 7)` | |

Now check your predictions.  `np.broadcast_shapes` applies the rules without making any arrays.

In [ ]:
def broadcast(shape_a, shape_b):
    try:
        return np.broadcast_shapes(shape_a, shape_b)
    except ValueError:
        return 'incompatible'

pairs = {
    'a': ((3,),         ()),
    'b': ((4, 3),       (3,)),
    'c': ((4, 3),       (4,)),
    'd': ((4, 1),       (3,)),
    'e': ((2, 1, 5),    (3, 1)),
    'f': ((2, 5),       (5, 2)),
    'g': ((1000, 2, 1), (2, 7)),
}

for name, (shape_a, shape_b) in pairs.items():
    print(name, shape_a, shape_b, '->', broadcast(shape_a, shape_b))

# A Times Table

Make a 12 by 12 times table, where `table[i, j]` is $(i+1) \times (j+1)$, out of the array `numbers` below.  Don't use any explicit loops!

In [ ]:
numbers = np.arange(1, 13)

table = ... # a (12, 12) times table, built from numbers with broadcasting

print(table)
print(table.shape)
print('7 × 8 =', table[6, 7])

# Centering Data

Here's some fake data: 6 trials of an experiment, each recording 4 sensors.  The rows are trials and the columns are sensors, so its shape is `(6, 4)`.

(Notice that `loc` is itself broadcast: each column gets its own average.)

In [ ]:
rng = np.random.default_rng(320)
data = rng.normal(loc=[10, 20, 30, 40], scale=1, size=(6, 4))
print(data.shape)
print(data)

First, subtract each *sensor's* average (taken over all trials) from that sensor's column, so that every column averages to 0.  `np.mean(data, axis=0)` averages over the trials.

In [ ]:
sensor_average = ... # the average of each sensor, over all the trials.  You can use np.mean's axis argument; check the documentation!
centered_sensors = ... # data with each sensor's average subtracted

print(sensor_average.shape)
print(np.mean(centered_sensors, axis=0))   # should be (very nearly) zeros

Now instead subtract each *trial's* average (taken over its four sensors) from that trial's row, so that every row averages to 0.  The obvious thing, `data - np.mean(data, axis=1)`, doesn't work.  Fix it using `np.newaxis`.

In [ ]:
trial_average = np.mean(data, axis=1)
centered_trials = data - trial_average # raises a shape-incompatibility error!

print(trial_average.shape)
print(np.mean(centered_trials, axis=1))   # should be (very nearly) zeros

**Why does `data - np.mean(data, axis=1)` fail, and why did subtracting the sensor averages work without any fix?**

# Every Distance at Once

Here are 50 random points in the plane, one per row, so `points` has shape `(50, 2)`.

In [ ]:
N = 50
points = rng.uniform(0, 1, size=(N, 2))
print(points.shape)

Compute the *displacement* from every point to every other point, `displacement[i, j] = points[j] - points[i]`, as one array of shape `(50, 50, 2)`.  Then compute the distance between every pair of points
\begin{align}
    \texttt{distance[i,j]} = \sqrt{\texttt{displacement[i,j,0]}^2 + \texttt{displacement[i,j,1]}^2}
\end{align}
with shape `(50, 50)`.  No loops!  Just broadcasting (and `np.newaxis` when needed)!

In [ ]:
displacement = ... # shape (N, N, 2): displacement[i, j] = points[j] - points[i]
distance = ... # shape (N, N): the length of each displacement

print(displacement.shape)
print(distance.shape)

Let's check against the slow way, with two loops.  This should print `True`.

In [ ]:
slow = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        slow[i, j] = np.sqrt(np.sum((points[j] - points[i])**2))

print(np.allclose(distance, slow))

What's the smallest distance between two *different* points?  Be careful: `np.min(distance)` is 0 because the distance from any point to itself is 0, so the diagonal of the distance matrix is all zeros.

In [ ]:
np.diag(distance)

Find a way around this issue (try using [`np.where`](https://numpy.org/doc/stable/reference/generated/numpy.where.html#numpy.where), [`np.eye`](https://numpy.org/doc/stable/reference/generated/numpy.eye.html#numpy.eye), and [`np.inf`](https://numpy.org/doc/stable/reference/constants.html#numpy.inf) (those are links to the documentation)).

In [ ]:
closest = ... # the smallest distance between two different points

print(closest)

# A Function on a Grid

Evaluate
\begin{align}
    z(x, y) &= \sin(\pi x) \cos(\pi y)
\end{align}
at every combination of the `x` and `y` below, as one array `z` with `z[i, j]` $= z(x_j, y_i)$.  That's the arrangement `plt.contourf(x, y, z)` expects: rows go with $y$ and columns go with $x$.

In [ ]:
x = np.linspace(-2, 2, 201)
y = np.linspace(-1, 1, 101)

z = ... # shape (len(y), len(x)), built with broadcasting

print(z.shape)   # should be (101, 201)

fig, ax = plt.subplots(figsize=(10, 5))
contours = ax.contourf(x, y, z, levels=20, cmap='RdBu_r')
fig.colorbar(contours)
ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('y')